# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [1]:
import numpy as np
import pandas as pd

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use.

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text.

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available.

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

Download new dataset for this excercise.

In [5]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("sh0416/ag_news")  # train/test
ag_train = pd.DataFrame(ds["train"])

# Create a single text field like the lab expects
ag_train["text"] = ag_train["title"].fillna("") + " " + ag_train["description"].fillna("")

# Human-readable topics
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "SciTech"}
ag_train["topic"] = ag_train["label"].map(label_map).fillna("Unknown")

# Keep it clean and consistent
ag_train = ag_train[["text", "topic"]]

# Save CSV in the working directory
ag_path = "ag_news.csv"
ag_train.to_csv(ag_path, index=False)

print("Saved:", ag_path)
ag_train.head()

KeyboardInterrupt: 

In [2]:
# news = pd.read_csv('/content/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT="title"
# TOPIC="topic"

# news = pd.read_csv("bbc_news.csv")
# MAX_NEWS = 1000
# DOCUMENT="description"
# TOPIC="title"

#news = pd.read_csv('/kaggle/input/mit-ai-news-published-till-2023/articles.csv')
#MAX_NEWS = 100
#DOCUMENT="Article Body"
#TOPIC="Article Header"

# use ag news dataset from hugging face
news = pd.read_csv("/content/ag_news.csv")
MAX_NEWS = 1000
DOCUMENT = "text"
TOPIC = "topic"

ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [3]:
news

,text,topic
0,Wall St. Bears Claw Back Into the Black (Reute...,SciTech
1,Carlyle Looks Toward Commercial Aerospace (Reu...,SciTech
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,SciTech
3,Iraq Halts Oil Exports from Main Southern Pipe...,SciTech
4,"Oil prices soar to all-time record, posing new...",SciTech
...,...,...
119995,Pakistan's Musharraf Says Won't Quit as Army C...,Sports
119996,Renteria signing a top-shelf deal Red Sox gene...,Business
119997,Saban not going to Dolphins yet The Miami Dolp...,Business
119998,Today's NFL games PITTSBURGH at NY GIANTS Time...,Business


In [4]:
news["id"] = news.index.astype(str)
news.head()

,text,topic,id
0,Wall St. Bears Claw Back Into the Black (Reute...,SciTech,0
1,Carlyle Looks Toward Commercial Aerospace (Reu...,SciTech,1
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,SciTech,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,SciTech,3
4,"Oil prices soar to all-time record, posing new...",SciTech,4


In [5]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)

## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database.

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior.

In [6]:
!pip install chromadb

In [7]:
import chromadb
from chromadb.config import Settings

Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance.

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that.

In [8]:
chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it.

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [9]:
collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)


It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***.
* In the **document** we store the big text, it's a different column in each Dataset.
* In **metadatas**, we can informa a list of topics.
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS.


In [10]:

collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)

In [11]:
results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

{'ids': [['id51', 'id320', 'id846', 'id311', 'id957', 'id714', 'id318', 'id845', 'id326', 'id390']], 'embeddings': None, 'documents': [["Delightful Dell The company's results show that it's not grim all over tech world. Just all of it that isn't Dell.", "Lenovo revenue grows, but problems persist China's largest PC maker sees surge in profits but loses market share to multinational companies.", "CompUSA Stores to Sell Gateway PCs  NEW YORK (Reuters) - Gateway Inc. &lt;GTW.N&gt;, the computer  maker that recently shuttered its chain of 188 stores, will  begin selling its desktops at CompUSA Inc.'s outlets later this  week, the two companies said on Monday.", "Briefly: HP partners with 3G gear maker roundup Plus: AMD starts shipping 90-nanometer chips...ABC.com to air on Real's SuperPass...Lenovo revenue grows, but problems persist.", "'SP2' a Must For XP Users (washingtonpost.com) washingtonpost.com - To get an idea of how Windows got to be such a mess, think of it as a house that was b

## Vector MAP

In [12]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [13]:

getado = collection.get(ids="id141",
                       include=["documents", "embeddings"])


In [14]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
#word_vectors

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents.

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search.


## Loading the model and creating the prompt

---


TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models.

We are importing:
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification.

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It have 3billion paramaters, more than enough for our sample, and works much better than GPT2.

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  


In [21]:
# # Update transformers and huggingface_hub libraries to ensure compatibility with Hugging Face models
# !pip install --upgrade transformers huggingface_hub

# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# model_id = "gpt2"    # Dolly-v2 was originally recommended by Databricks, but it could not be loaded in this environment due to availability or compatibility issues, so GPT-2 is used as an alternative.

# # It's good practice to add `trust_remote_code=True` when loading models that might have custom code
# # or if experiencing issues, although not strictly required for all models.
# # For Dolly-v2 models, `trust_remote_code=True` is often recommended.
# tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
# lm_model = AutoModelForCausalLM.from_pretrained(model_id)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 49.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Model: Nanbeige4.1-3B

In [15]:
# Update transformers + huggingface_hub for compatibility
!pip -q install --upgrade transformers huggingface_hub accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "Nanbeige/Nanbeige4.1-3B"

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_fast=False,  # safer with some custom tokenizers
)

lm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True, # ensures our model runs its biult in configuration
    torch_dtype=torch.float16,
    device_map="auto",
)

# Create a generation pipeline (similar to the lab's approach)
gen_pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The next step is to initialize the pipeline using the objects created above.

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [16]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512, # Double 256 for this lab anyway as this is a bigger model
    device_map="auto",
)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user.

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**.

We only need to join the two parts together to create the prompt that we are going to send to the model.

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part.

In [17]:
question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"""
Relevant context:
{context}

Question:
{question}

Answer:
"""
prompt_template

"\nRelevant context:\n#Delightful Dell The company's results show that it's not grim all over tech world. Just all of it that isn't Dell. #Lenovo revenue grows, but problems persist China's largest PC maker sees surge in profits but loses market share to multinational companies. #CompUSA Stores to Sell Gateway PCs  NEW YORK (Reuters) - Gateway Inc. &lt;GTW.N&gt;, the computer  maker that recently shuttered its chain of 188 stores, will  begin selling its desktops at CompUSA Inc.'s outlets later this  week, the two companies said on Monday. #Briefly: HP partners with 3G gear maker roundup Plus: AMD starts shipping 90-nanometer chips...ABC.com to air on Real's SuperPass...Lenovo revenue grows, but problems persist. #'SP2' a Must For XP Users (washingtonpost.com) washingtonpost.com - To get an idea of how Windows got to be such a mess, think of it as a house that was built on an island in the middle of a lake, deep in the countryside. #Gateway lands on another retailer's shelves Company i

Now all that remains is to send the prompt to the model and wait for its response!


In [18]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Relevant context:
#Delightful Dell The company's results show that it's not grim all over tech world. Just all of it that isn't Dell. #Lenovo revenue grows, but problems persist China's largest PC maker sees surge in profits but loses market share to multinational companies. #CompUSA Stores to Sell Gateway PCs  NEW YORK (Reuters) - Gateway Inc. &lt;GTW.N&gt;, the computer  maker that recently shuttered its chain of 188 stores, will  begin selling its desktops at CompUSA Inc.'s outlets later this  week, the two companies said on Monday. #Briefly: HP partners with 3G gear maker roundup Plus: AMD starts shipping 90-nanometer chips...ABC.com to air on Real's SuperPass...Lenovo revenue grows, but problems persist. #'SP2' a Must For XP Users (washingtonpost.com) washingtonpost.com - To get an idea of how Windows got to be such a mess, think of it as a house that was built on an island in the middle of a lake, deep in the countryside. #Gateway lands on another retailer's shelves Company is s

## Excercise Run

### 1) **Strict**

In [44]:
import re
import html

def clean_context_docs(docs):
    cleaned = []
    for d in docs:
        if d is None:
            continue
        s = str(d)

        # Fix common dataset artifacts
        s = html.unescape(s)                  # &lt;GTW.N&gt; -> <GTW.N>
        s = s.replace("\\n", " ").replace("\\t", " ")
        s = s.replace("\\\\", "\\")           # collapse double backslashes

        # Normalize whitespace
        s = re.sub(r"\s+", " ", s).strip()

        if s:
            cleaned.append(s)

    return cleaned

def run_generation(prompt_text, gen_cap=200):
    input_len = len(tokenizer(prompt_text, return_tensors="pt")["input_ids"][0])
    max_length = input_len + gen_cap

    out = pipe(
        prompt_text,
        max_length=max_length,
        return_full_text=False,
        do_sample=False,
    )
    return out[0]["generated_text"].strip()

In [45]:
STRICT_PROMPT_TEMPLATE = """You are a grounded Q&A assistant.

RULES:
- Use ONLY the information in the CONTEXT.
- If the answer is not explicitly supported by the CONTEXT, respond exactly:
  "I don't know based on the provided context."
- Do not add outside knowledge.
- Keep the answer to 1-3 sentences.
- Do not explain your reasoning.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

In [46]:
def generate_strict_answer(question: str, context: str) -> str:
    prompt = STRICT_PROMPT_TEMPLATE.format(context=context, question=question)

    # Same call pattern as the lab
    lm_response = pipe(prompt)
    raw = lm_response[0]["generated_text"]

    # Extract only what the model added (not the whole prompt)
    if "ANSWER:" in raw:
        text = raw.split("ANSWER:", 1)[1].strip()
    else:
        text = raw[len(prompt):].strip() if raw.startswith(prompt) else raw.strip()

    # Strip <think> blocks if they appear
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    if "<think>" in text:
        text = text.split("<think>", 1)[0].strip()

    # Light enforcement: keep it short (first ~3 sentences)
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return " ".join(sentences[:3]).strip()

In [47]:
question = "Can I buy a Toshiba laptop?"

docs_clean = clean_context_docs(results["documents"][0])
context = " ".join([f"#{d}" for d in docs_clean])
context = context[:5120]

prompt_template_1 = STRICT_PROMPT_TEMPLATE.format(context=context, question=question)

strict_answer = run_generation(prompt_template_1, gen_cap=200)
print(strict_answer)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=713) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
We are given a context that is a collection of news snippets about various tech companies and events. The question is: "Can I buy a Toshiba laptop?"

First, I need to check the context to see if there is any information about Toshiba laptops. The context provided includes:
- Dell results
- Lenovo revenue
- China's PC maker
- CompUSA Stores to Sell Gateway PCs
- Gateway alumnus resurfaces at HP
- Dell Unveils Inexpensive Projector
- Other snippets about HP, AMD, etc.

I need to look for any mention of Toshiba in the context. Let me scan through the context:

The context starts with:
"#Delightful Dell The company's results show that it's not grim all over tech world. Just all of it that isn't Dell. #Lenovo revenue grows, but problems persist China's largest PC maker sees surge in profits but loses market share to multinational companies. #CompUSA Stores to Sell Gateway PCs NEW YORK (Reuters) - Gateway Inc. <GTW.N>, the computer maker that recently shuttered its chain of 188 store

### 2) **Evidence**

In [48]:
EVIDENCE_PROMPT_TEMPLATE = """You are a grounded Q&A assistant.

RULES:
- Use ONLY the CONTEXT.
- If unsupported, respond exactly:
  "I don't know based on the provided context."
- If supported, provide:
  1) Answer (1-2 sentences)
  2) Evidence: a short phrase copied from the context (max 20 words)

CONTEXT:
{context}

QUESTION:
{question}

OUTPUT:
"""

In [49]:
def generate_evidence_answer(question: str, context: str) -> str:
    prompt = EVIDENCE_PROMPT_TEMPLATE.format(context=context, question=question)

    # Same call pattern as the lab
    lm_response = pipe(prompt)
    raw = lm_response[0]["generated_text"]

    # Extract only what the model added
    if "OUTPUT:" in raw:
        text = raw.split("OUTPUT:", 1)[1].strip()
    else:
        text = raw[len(prompt):].strip() if raw.startswith(prompt) else raw.strip()

    # Strip <think> blocks if they appear
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    if "<think>" in text:
        text = text.split("<think>", 1)[0].strip()

    return text

In [50]:
question = "Can I buy a Toshiba laptop?"

docs_clean = clean_context_docs(results["documents"][0])
context = " ".join([f"#{d}" for d in docs_clean])
context = context[:5120]

prompt_template_2 = EVIDENCE_PROMPT_TEMPLATE.format(context=context, question=question)

evidence_answer = run_generation(prompt_template_2, gen_cap=240)
print(evidence_answer)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=512) and `max_length`(=749) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
We are given a context that is a collection of news snippets and headlines. The question is: "Can I buy a Toshiba laptop?"

First, I need to check the context to see if there is any information about Toshiba laptops being available for purchase.

Looking through the context:

- The context mentions Dell, Lenovo, HP, Gateway, CompUSA, BEA, Cisco, etc.
- There is a mention of "Gateway alumnus resurfaces at HP" but that is about an executive, not about Toshiba.
- There is a mention of "Dell Unveils Inexpensive Projector" but that is about Dell, not Toshiba.
- There is no mention of Toshiba in the context at all.

The context provided does not contain any information about Toshiba laptops, whether they are available for purchase, in stock, or any related details.

Therefore, based on the provided context, I cannot answer the question about buying a Toshiba laptop.

According to the rules:
- If unsupported, respond exactly: "I don't know based on the provided context."

So, the answ

## Lab Reflection — Retrieval + Prompting (Strict vs Evidence Answers)

### What we built
- Loaded a news dataset into a **ChromaDB vector store** and queried it to retrieve **relevant context passages** for a user question.
- Used the retrieved passages to construct a single **“context + question” prompt** (RAG-style prompt).
- Ran two answer strategies using the same retrieved context:
  - **`generate_strict_answer(question, context)`** → must answer *only if* supported by the context, otherwise returns:  
    **"I don't know based on the provided context."**
  - **`generate_evidence_answer(question, context)`** → same grounding rule, but when supported it must return an answer plus a short **evidence quote** from the context.

### Key learning points
- **Retrieval improves grounding:** the model should rely on *what we retrieved* instead of free-associating from general knowledge.
- **Prompt rules are a control layer:** strict formatting + refusal rules reduce hallucinations, especially when the context doesn’t contain the answer.
- **Evidence-based answers are stronger:** requiring a short quote forces the response to be tied to something actually present in the retrieved text.
- **Real-world behavior isn’t perfect:** even with strong prompts, models can still add extra text (warnings / reasoning / verbosity). This is why real systems often combine:
  - stronger prompting,
  - output validation,
  - and optional post-processing.

### Outcome (example)
- For a question not supported by the retrieved context (“Can I buy a Toshiba laptop?”), the model correctly fell back to:  
  **"I don't know based on the provided context."**  
  demonstrating the core “grounded QA” concept.

### Final takeaway
**RAG = Retrieve → Build Context Prompt → Answer with constraints.**  
Adding a **strict refusal rule** and an **evidence requirement** is a practical way to make chatbot outputs more reliable and explainable.